### Treinamento e Teste do Modelo Escolhido, Teste de Predição e Salvamento do Modelo

### Importação das bibliotecas

In [23]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

### Carregamento e pré-processamento dos dados

In [24]:
raw_data = pd.read_excel('../datasets/ENB2012_data.xlsx')

#### Como testado, removeremos as variáveis X4 e X5 para os melhores resultados

In [25]:
new_data = raw_data.drop(['X4', 'X5'], axis='columns')

###  Separação de features e target

In [26]:
X = new_data.drop(['Y1', 'Y2'], axis=1)
y = new_data[['Y1', 'Y2']]

### Divisão em treino e teste

In [27]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Escalonamento

In [28]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#### Treinamento do Modelo

In [29]:
model = XGBRegressor(random_state=42, verbosity=0)

# Treinamento do modelo
model.fit(X_train_scaled, y_train)

# Previsões
y_pred = model.predict(X_test_scaled)

# Avaliação
mse_avg = mean_squared_error(y_test, y_pred, multioutput='uniform_average')
r2_avg = r2_score(y_test, y_pred, multioutput='uniform_average')

print(f"Mean Squared Error: {mse_avg:.4f}")
print(f"R² Score: {r2_avg:.4f}")

Mean Squared Error: 0.4077
R² Score: 0.9957


#### Otimização de hiperparâmetros

In [15]:
# Definir os parâmetros para busca
param_grid = {
    'n_estimators': [25, 50, 100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3],
    'max_depth': [3, 4, 5, 6, 7, 8],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

# Criar o modelo para GridSearch
grid_search = GridSearchCV(
    estimator=XGBRegressor(objective='reg:squarederror', random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

# Executar a busca
grid_search.fit(X_train_scaled, y_train)

# Melhores parâmetros encontrados
print("\nMelhores parâmetros encontrados:")
print(grid_search.best_params_)

# Melhor modelo
best_model = grid_search.best_estimator_

# Fazer previsões com o melhor modelo
y_pred_best = best_model.predict(X_test_scaled)

# Avaliar o melhor modelo
mse_best = mean_squared_error(y_test, y_pred_best)
r2_best = r2_score(y_test, y_pred_best)

print(f"\nMelhor Mean Squared Error: {mse_best:.4f}")
print(f"Melhor R² Score: {r2_best:.4f}")

Fitting 5 folds for each of 1350 candidates, totalling 6750 fits

Melhores parâmetros encontrados:
{'colsample_bytree': 0.9, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 300, 'subsample': 0.8}

Melhor Mean Squared Error: 0.5019
Melhor R² Score: 0.9947


In [30]:
# Salvar o modelo
joblib.dump(model, '../DjangoAPI/ml_models/xgboost_regressor_model.pkl')

# Salvar o scaler
joblib.dump(scaler, '../DjangoAPI/ml_models/xgboost_scaler.pkl')


['../DjangoAPI/ml_models/xgboost_scaler.pkl']

#### Uma gridsearch foi realizada para buscar melhores parâmetros para o XGBoostRegressor, mas o melhor fit continua sendo dos parâmetros padrão.

#### Ordens de grandeza das Variáveis

In [31]:
minimos = new_data.min()
maximos = new_data.max()

print("Valores mínimos:\n", minimos)
print("\nValores máximos:\n", maximos)

Valores mínimos:
 X1      0.62
X2    514.50
X3    245.00
X6      2.00
X7      0.00
X8      0.00
Y1      6.01
Y2     10.90
dtype: float64

Valores máximos:
 X1      0.98
X2    808.50
X3    416.50
X6      5.00
X7      0.40
X8      5.00
Y1     43.10
Y2     48.03
dtype: float64


### Modelo de Script de Predição

In [32]:
# Carregar o modelo e o scaler
model = joblib.load('../DjangoAPI/ml_models/xgboost_regressor_model.pkl')
scaler = joblib.load('../DjangoAPI/ml_models/xgboost_scaler.pkl')

# Dataframe para predição
df_predict = pd.DataFrame(columns=['X1', 'X2', 'X3', 'X6', 'X7', 'X8'])

# Novos dados
new_predict = [[0.73, 700.0, 300.0, 2, 0.3, 3]]

# Adicionar no dataframe
df_predict.loc[0] = new_predict[0]

# Aplicar o mesmo scaler usado no treino
new_predict_scaled = scaler.transform(df_predict)

# Fazer a predição
prediction = model.predict(new_predict_scaled)
print("Predição:", prediction)

Predição: [[12.241612 14.804065]]
